In [1]:
# Import necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Load the dataset
file_path = 'C:/Users/nosao/Desktop/Maxwell-Text Classification/Target Response/data/target_planning_resource.csv' #Replace with actual file path
df = pd.read_csv(file_path)
df.columns = df.columns.str.strip()  # Clean column names

# Clean missing values
df = df.dropna(subset=['Job description', 'Question 22', 'Question 23', 'Question 24', 'Question 25', 'Question 26'])

# Convert target variables to strings
df['Question 22'] = df['Question 22'].astype(str)
df['Question 23'] = df['Question 23'].astype(str)
df['Question 24'] = df['Question 24'].astype(str)
df['Question 25'] = df['Question 25'].astype(str)
df['Question 26'] = df['Question 26'].astype(str)


# Extract job descriptions and target variables
X = df['Job description'].astype(str)
y_q22 = df['Question 22']
y_q23 = df['Question 23']
y_q24 = df['Question 24']
y_q25 = df['Question 25']
y_q26 = df['Question 26']

# Split data into training and test sets for each question
X_train, X_test, y_train_22, y_test_22 = train_test_split(X, y_q22, test_size=0.2, random_state=42)
X_train_q23, X_test_q23, y_train_q23, y_test_q23 = train_test_split(X, y_q23, test_size=0.2, random_state=42)
X_train_q24, X_test_q24, y_train_q24, y_test_q24 = train_test_split(X, y_q24, test_size=0.2, random_state=42)
X_train_q25, X_test_q25, y_train_q25, y_test_q25 = train_test_split(X, y_q25, test_size=0.2, random_state=42)
X_train_q26, X_test_q26, y_train_q26, y_test_q26 = train_test_split(X, y_q26, test_size=0.2, random_state=42)

# Define Logistic Regression with class_weight='balanced' to handle class imbalance
logreg_pipeline = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000, class_weight='balanced'))

# Hyperparameter tuning grid
param_grid_logreg = {
    'logisticregression__C': [0.01, 0.1, 1, 10],
    'tfidfvectorizer__ngram_range': [(1, 1), (1, 2), (1, 3)],
    'tfidfvectorizer__max_df': [0.85, 0.9, 0.95],
    'tfidfvectorizer__min_df': [1, 5],
    'tfidfvectorizer__use_idf': [True, False],
    'tfidfvectorizer__sublinear_tf': [True, False]
}

# Hyperparameter tuning with StratifiedKFold to ensure balanced class representation in cross-validation
cv = StratifiedKFold(n_splits=5)

# Hyperparameter tuning for Logistic Regression for each question
grid_logreg_q22 = GridSearchCV(logreg_pipeline, param_grid_logreg, cv=cv, scoring='accuracy', n_jobs=-1)
grid_logreg_q22.fit(X_train, y_train_22)

grid_logreg_q23 = GridSearchCV(logreg_pipeline, param_grid_logreg, cv=cv, scoring='accuracy', n_jobs=-1)
grid_logreg_q23.fit(X_train_q23, y_train_q23)

grid_logreg_q24 = GridSearchCV(logreg_pipeline, param_grid_logreg, cv=cv, scoring='accuracy', n_jobs=-1)
grid_logreg_q24.fit(X_train_q24, y_train_q24)

grid_logreg_q25 = GridSearchCV(logreg_pipeline, param_grid_logreg, cv=cv, scoring='accuracy', n_jobs=-1)
grid_logreg_q25.fit(X_train_q25, y_train_q25)

grid_logreg_q26 = GridSearchCV(logreg_pipeline, param_grid_logreg, cv=cv, scoring='accuracy', n_jobs=-1)
grid_logreg_q26.fit(X_train_q26, y_train_q26)

# Use the best Logistic Regression models for each question
best_logreg_model_q22 = grid_logreg_q22.best_estimator_
best_logreg_model_q23 = grid_logreg_q23.best_estimator_
best_logreg_model_q24 = grid_logreg_q24.best_estimator_
best_logreg_model_q25 = grid_logreg_q25.best_estimator_
best_logreg_model_q26 = grid_logreg_q26.best_estimator_

# Function to display metrics
def display_metrics(y_true, y_pred, question_num):
    print(f"Metrics for Question {question_num}")
    print(classification_report(y_true, y_pred))
    print(f"Accuracy: {accuracy_score(y_true, y_pred)}\n")

# Predict on the full test set for each question and display metrics
y_pred_q22 = best_logreg_model_q22.predict(X_test)
display_metrics(y_test_22, y_pred_q22, 22)

y_pred_q23 = best_logreg_model_q23.predict(X_test_q23)
display_metrics(y_test_q23, y_pred_q23, 23)

y_pred_q24 = best_logreg_model_q24.predict(X_test_q24)
display_metrics(y_test_q24, y_pred_q24, 24)

y_pred_q25 = best_logreg_model_q25.predict(X_test_q25)
display_metrics(y_test_q25, y_pred_q25, 25)

y_pred_q26 = best_logreg_model_q26.predict(X_test_q26)
display_metrics(y_test_q26, y_pred_q26, 26)

# Function to make predictions on a new job description
def predict_for_new_job_description(job_description):
    # Ensure the input is a string
    job_description = [job_description]

    # Predict for each question using the best model
    pred_q22 = best_logreg_model_q22.predict(job_description)[0]
    pred_q23 = best_logreg_model_q23.predict(job_description)[0]
    pred_q24 = best_logreg_model_q24.predict(job_description)[0]
    pred_q25 = best_logreg_model_q25.predict(job_description)[0]
    pred_q26 = best_logreg_model_q26.predict(job_description)[0]

    # Display or return the results
    print("Predictions for the new job description:")
    print(f"Question 22: {pred_q22}")
    print(f"Question 23: {pred_q23}")
    print(f"Question 24: {pred_q24}")
    print(f"Question 25: {pred_q25}")
    print(f"Question 26: {pred_q26}")

# Example: Enter a new job description
new_description = """
Purpose of the Role: To provide an effective Joinery resource to ensure the University
fabric is efficiently maintained on a day-to-day basis including undertaking Project works. To
ensure the effective interaction of Estate and Facilities services with other services.

Responsible to: Estates Team Leader

Main Duties and Responsibilities:
1. To provide all forms of Joinery duties and tasks in which you are competent within the
University Estate possessing at least five years of trade experience. Working with the team
across various other construction trades.
2. To be responsible for day-to-day breakdown and reactive maintenance.
3. To participate in the Maintenance call-out rota team.
4. To be responsible for working to and delivering cyclical maintenance works ensuring
certain activities are carried out as per the PPM regime.
5. To manage the fire door programme focussing on the Inspection, maintenance (ART
accepted repair techniques), and repair to achieve a compliant campus. This will involve
a good theoretical knowledge of the standards and regulations.
6. To be responsible for the Fire door dashboard, upkeep, and maintenance of the system.
To provide information on defects and repair (analyzing reports), accepted repair
techniques, and input for Projects.
7. To maintain Fire door records on CAFM and in line with the Fire Safety Regulations
(2023). Records must be kept.
8. Identify hazards, defects, and the need for adjustment or repair; to ensure compliance
with agreed codes, law, working practices, and health and safety whilst carrying out your
duties.
9. To provide support and guidance to Contractors engaged in Fire door campus works and
act as a focal point ensuring a fully compliant Fire door install is delivered to the Estate.
10. To manage quantities required to complete each task and manage material stocks and
ordering process.
11. To be responsible for ensuring all tools and equipment are maintained in good working
order and ready for use including power tools within the Estate.
12. To ensure all University fixtures, fittings, furniture, doors, locks, flooring, and other
Joinery items are efficiently maintained, repaired, constructed, or replaced, working
closely with Estates, Facilities Managers, and Estates Team Leader to achieve.
13. To ensure works are delivered in compliance with documented risk assessments and Method
statements, and responsible for the production and review of role-specific risk
assessments.
14. To be responsible for a high standard of conduct always working in a safe and
professional manner reporting any health and safety-related issues to the Estates Team
Leader immediately.
15. To be responsible for working to and delivering all works and repairs in a manner that
ensures VFM and quality finishes are implemented and maintained.
16. To assist the team and organization with general duties over and above your core skills.
Promote, develop and expand the business of our organization generally meeting set
targets.
17. To aid and advise the other members of the University Estates & Facilities staff including
porters and grounds staff as required.
18. To adhere to all organization policies and procedures.
19. To be responsible for continued professional development ensuring the post holder is
conversant and aware of current regulations, legislation, and approved industry
standards to their job role. You will have a basic awareness of Asbestos, CDM,
health and safety regulations.
20. To undertake small projects as reasonably required of the job role and advise all Estates
teams to deliver solutions and cost reductions to all joinery works on campus.

General Duties:
21. To ensure the use of data complies with current regulations, particularly those relating to
GDPR.
22. To comply with all health, safety, and wellbeing policies and procedures at all times and to
take responsibility for promoting and safeguarding the welfare and protection of others.
23. To advocate, promote, and advance equity and social justice within your work.
24. To carry out other duties, commensurate with the grade of the post, as may reasonably be
directed by your line manager after due consultation.
"""



c:\Users\nosao\Desktop\spacyProj\spacy_venv\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\nosao\Desktop\spacyProj\spacy_venv\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\nosao\Desktop\spacyProj\spacy_venv\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\nosao\Desktop\spacyProj\spacy_venv\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


Metrics for Question 22
              precision    recall  f1-score   support

           A       0.00      0.00      0.00         1
           B       0.00      0.00      0.00         2
           C       0.67      0.67      0.67         3
           D       0.81      1.00      0.90        13

    accuracy                           0.79        19
   macro avg       0.37      0.42      0.39        19
weighted avg       0.66      0.79      0.72        19

Accuracy: 0.7894736842105263

Metrics for Question 23
              precision    recall  f1-score   support

           A       0.00      0.00      0.00         2
           B       0.60      0.60      0.60         5
           C       0.50      0.67      0.57         6
           D       0.17      0.17      0.17         6

    accuracy                           0.42        19
   macro avg       0.32      0.36      0.33        19
weighted avg       0.37      0.42      0.39        19

Accuracy: 0.42105263157894735

Metrics for Question 

c:\Users\nosao\Desktop\spacyProj\spacy_venv\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\nosao\Desktop\spacyProj\spacy_venv\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\nosao\Desktop\spacyProj\spacy_venv\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} i